# Module 05: Feast (Feature Store)

## What Problem Does a Feature Store Solve?

Imagine you've computed creator embeddings and cluster assignments as part of your training pipeline. Now you need those features in two places:

1. **Training:** Join features with historical labels to train a model. You need features *as they were at a specific point in time* — not the current value, but the value when the training example was observed.
2. **Serving:** The chatbot needs a creator's embedding in <10ms to do RAG. You can't recompute a 1024-dim embedding on every request.

Without a feature store:
- Your training pipeline writes features to one place (S3 Parquet)
- Your serving pipeline reads from Redis, maintaining a separate sync job
- These diverge. Training uses different feature logic than serving. **Training-serving skew** causes silent accuracy drops.

**Feast solves this** by being the single source of truth:
- **Offline store** (S3/Parquet) → training, batch scoring
- **Online store** (Redis) → real-time serving
- Same feature definitions, same retrieval API, guaranteed consistency

## The Offline/Online Split

```
Offline Store (S3 Parquet)          Online Store (Redis)
├── All historical features          ├── Only latest feature values
├── Partitioned by date              ├── Key: entity_id
├── Used for training                ├── Used for real-time serving
└── Slow (seconds/minutes)           └── Fast (<5ms)
                    ↑
            feast materialize
            (push offline → online)
```

## Key Vocabulary

| Term | Meaning |
|---|---|
| **Entity** | The primary key — what you look features up by (e.g., `creator_id`) |
| **Feature View** | A group of related features with a data source and TTL |
| **Feature** | One column in a Feature View (e.g., `subscriber_count`) |
| **Feature Service** | A named collection of features from multiple views — what the serving layer requests |
| **Offline Store** | Historical feature storage (Parquet files in S3 or locally) |
| **Online Store** | Low-latency store for latest feature values (Redis, SQLite) |
| **Materialization** | The `feast materialize` command that pushes offline → online |
| **Point-in-time join** | Joining features to training data using historical values at the right timestamp |


## Setup

```bash
pip install feast pandas numpy
```

For this tutorial we use **local file** for offline store and **SQLite** for online store — no S3, no Redis needed. In the Galaxy project you'd swap these for Oracle Object Storage and Redis.

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Clean up from previous runs
if os.path.exists("./galaxy_feature_store"):
    shutil.rmtree("./galaxy_feature_store")
os.makedirs("./galaxy_feature_store/data", exist_ok=True)

print("Working directory cleaned")

---
## Exercise 1: Define the Feature Store

Feast needs a `feature_store.yaml` config file and Python files defining entities + feature views.

In [ ]:
# Write the Feast config file
feature_store_yaml = """
project: galaxy
registry: ./galaxy_feature_store/registry.db
provider: local
offline_store:
    type: file
online_store:
    type: sqlite
    path: ./galaxy_feature_store/online_store.db
entity_key_serialization_version: 2
"""

with open("./galaxy_feature_store/feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print("feature_store.yaml written")

In [ ]:
# In production, these are in features/entities.py and features/feature_views.py
# Here we define them inline

from feast import Entity, FeatureStore, FeatureView, Field, FileSource
from feast.types import Float32, Float64, Int64, String

# --- Entity: the primary key ---
# Everything in the feature store is looked up by creator_id
creator_entity = Entity(
    name="creator_id",
    description="Fandom page ID for a YouTube creator (e.g. fandom_12345)",
)

print("Entity defined:", creator_entity.name)

---
## Exercise 2: Create Feature Data and Define Feature Views

In [ ]:
# Create realistic feature data
now = datetime.utcnow()

# --- Creator profile features (from scraper output) ---
profile_data = pd.DataFrame({
    "creator_id": ["fandom_001", "fandom_002", "fandom_003", "fandom_004", "fandom_005"],
    "display_name": ["MrBeast", "PewDiePie", "Markiplier", "Kurzgesagt", "CGP Grey"],
    "cluster_id": [0, 1, 1, 2, 2],
    "cluster_name": ["Challenge", "Gaming Commentary", "Gaming Commentary", "Educational", "Educational"],
    "event_timestamp": [now] * 5,  # Feast requires this column
    "created_timestamp": [now] * 5,
})

# --- Embedding features (from training pipeline output) ---
np.random.seed(42)
embedding_data = pd.DataFrame({
    "creator_id": ["fandom_001", "fandom_002", "fandom_003", "fandom_004", "fandom_005"],
    "umap_x": np.random.uniform(-20, 20, 5).tolist(),
    "umap_y": np.random.uniform(-20, 20, 5).tolist(),
    "umap_z": np.random.uniform(-20, 20, 5).tolist(),
    "embedding_dim": [1024] * 5,  # The actual 1024-dim vector is stored in Weaviate, not Feast
    "event_timestamp": [now] * 5,
    "created_timestamp": [now] * 5,
})

# Save to Parquet (the offline store)
profile_data.to_parquet("./galaxy_feature_store/data/creator_profiles.parquet")
embedding_data.to_parquet("./galaxy_feature_store/data/creator_embeddings.parquet")

print("Profile data:")
print(profile_data[["creator_id", "display_name", "cluster_name"]].to_string(index=False))
print("\nEmbedding data:")
print(embedding_data[["creator_id", "umap_x", "umap_y", "umap_z"]].to_string(index=False))

In [ ]:
# Define Feature Views that point to the Parquet files

# Feature View 1: Creator profile
creator_profile_source = FileSource(
    path="./galaxy_feature_store/data/creator_profiles.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

creator_profile_fv = FeatureView(
    name="creator_profile",
    entities=[creator_entity],
    ttl=timedelta(days=7),  # online store entries expire after 7 days
    schema=[
        Field(name="display_name", dtype=String),
        Field(name="cluster_id", dtype=Int64),
        Field(name="cluster_name", dtype=String),
    ],
    source=creator_profile_source,
)

# Feature View 2: Embeddings (UMAP coordinates; actual 1024-dim vectors live in Weaviate)
creator_embedding_source = FileSource(
    path="./galaxy_feature_store/data/creator_embeddings.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

creator_embedding_fv = FeatureView(
    name="creator_embedding",
    entities=[creator_entity],
    ttl=timedelta(days=7),
    schema=[
        Field(name="umap_x", dtype=Float64),
        Field(name="umap_y", dtype=Float64),
        Field(name="umap_z", dtype=Float64),
        Field(name="embedding_dim", dtype=Int64),
    ],
    source=creator_embedding_source,
)

print("Feature views defined:", [creator_profile_fv.name, creator_embedding_fv.name])

---
## Exercise 3: Apply and Materialize

`feast apply` registers your entities and feature views.
`feast materialize-incremental` pushes recent offline data to the online store.

In Galaxy, these are called by Airflow tasks in the `training_pipeline_dag`.

In [ ]:
# Initialize the store and apply definitions
store = FeatureStore(repo_path="./galaxy_feature_store")

# Apply all objects to the registry
store.apply([creator_entity, creator_profile_fv, creator_embedding_fv])

print("Applied to registry:")
print("  Entities:", [e.name for e in store.list_entities()])
print("  Feature Views:", [fv.name for fv in store.list_feature_views()])

In [ ]:
# Materialize: push offline data to the online store (SQLite locally, Redis in production)
# In production: store.materialize_incremental(end_date=datetime.utcnow())
# Here we materialize everything from the past 30 days

store.materialize(
    start_date=now - timedelta(days=30),
    end_date=now + timedelta(seconds=1)  # slightly in future to include 'now' timestamps
)

print("\nMaterialization complete — features are now in the online store")

---
## Exercise 4: Retrieve Online Features

This is what the chatbot calls in production. Entity key → feature values in <5ms.

In [ ]:
# Retrieve features for a single creator (e.g., a chatbot query for MrBeast)
entity_rows = [{"creator_id": "fandom_001"}]  # MrBeast

online_features = store.get_online_features(
    features=[
        "creator_profile:display_name",
        "creator_profile:cluster_id",
        "creator_profile:cluster_name",
        "creator_embedding:umap_x",
        "creator_embedding:umap_y",
        "creator_embedding:umap_z",
    ],
    entity_rows=entity_rows,
).to_dict()

print("Online features for fandom_001 (MrBeast):")
for key, val in online_features.items():
    print(f"  {key}: {val[0]}")

In [ ]:
# Batch retrieval — multiple creators at once
entity_rows = [
    {"creator_id": "fandom_001"},
    {"creator_id": "fandom_002"},
    {"creator_id": "fandom_004"},
]

online_features = store.get_online_features(
    features=["creator_profile:display_name", "creator_profile:cluster_name"],
    entity_rows=entity_rows,
).to_dict()

print("Batch retrieval:")
for i in range(len(entity_rows)):
    cid = online_features["creator_id"][i]
    name = online_features["creator_profile__display_name"][i]
    cluster = online_features["creator_profile__cluster_name"][i]
    print(f"  {cid}: {name} → {cluster}")

---
## Exercise 5: Offline Features for Training

During training, you need features as they were at a specific time — not today's values. This is **point-in-time correctness**, the feature store's killer feature for ML.

Imagine you're training a model to predict creator growth. Your training label is "did the subscriber count increase in the next 30 days?". You need features *at the time the label was measured*, not current values (which would leak future information).

In [ ]:
# Entity DataFrame: the training examples with their timestamps
# Feast will join the feature values that were valid AT each timestamp
entity_df = pd.DataFrame({
    "creator_id": ["fandom_001", "fandom_002", "fandom_003"],
    "event_timestamp": [
        now - timedelta(days=5),   # 5 days ago
        now - timedelta(days=3),   # 3 days ago
        now - timedelta(days=1),   # yesterday
    ]
})

# Get historical features (point-in-time join against offline store)
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "creator_profile:cluster_id",
        "creator_profile:cluster_name",
        "creator_embedding:umap_x",
        "creator_embedding:umap_y",
    ]
).to_df()

print("Training features (point-in-time):")
print(training_df.to_string(index=False))
print("\nThese are the feature values at each event_timestamp, not current values.")

---
## Exercise 6: Feature Service

A **Feature Service** bundles a set of features from multiple views under a single name. It's the contract between your feature store and your serving code — the chatbot requests a named service, not individual features.

In [ ]:
from feast import FeatureService

# Define a feature service for the chatbot
chatbot_feature_service = FeatureService(
    name="chatbot_features",
    features=[
        creator_profile_fv[["display_name", "cluster_id", "cluster_name"]],
        creator_embedding_fv[["umap_x", "umap_y", "umap_z"]],
    ]
)

store.apply([chatbot_feature_service])

# Now retrieve using the service name (cleaner API for serving code)
features = store.get_online_features(
    features=store.get_feature_service("chatbot_features"),
    entity_rows=[{"creator_id": "fandom_001"}]
).to_dict()

print("Chatbot feature service retrieval for MrBeast:")
for k, v in features.items():
    print(f"  {k}: {v[0]}")

---
## How Feast Maps to the Galaxy Project

```
training_pipeline_dag (Airflow, runs locally)
│
├── SageMaker / GPU Job → produces embeddings.parquet
│
├── feast materialize-incremental task
│       Offline: Oracle Object Storage (s3-compatible)
│       Online: Redis (in Oracle k3s)
│       ↓
│   Features in Redis: ready for serving
│
└── Oracle reload webhook → Weaviate upsert

FastAPI chatbot (Oracle k3s)
│
├── Entity lookup: feast.get_online_features([{creator_id: X}])
│   → returns cluster_name, display_name, umap coords in <5ms
│
├── Weaviate near_vector search → top 10 similar creators
│
└── Build prompt → Groq API → stream response
```

**Production config** (features/feature_store.yaml):
```yaml
project: galaxy
provider: local
offline_store:
    type: s3
    bucket: galaxy-features
    s3_endpoint_override: https://objectstorage.us-phoenix-1.oraclecloud.com  # Oracle S3-compat
online_store:
    type: redis
    connection_string: redis-service:6379
registry: s3://galaxy-features/registry/registry.db
```

## Summary

| Action | Code / Command |
|---|---|
| Initialize store | `store = FeatureStore(repo_path="./")` |
| Register definitions | `store.apply([entity, feature_view, feature_service])` |
| Materialize offline→online | `store.materialize_incremental(end_date=datetime.utcnow())` |
| Get online features | `store.get_online_features(features=[...], entity_rows=[{...}])` |
| Get historical features | `store.get_historical_features(entity_df=..., features=[...])` |